# Entraînement et Performances des Modèles

Ce notebook rassemble de manière interactive l'entraînement des modèles de classification (pour prédire la criticité) et l'évaluation des modèles de séries temporelles (pour prévoir le trafic normal).

In [3]:
import sys
from pathlib import Path

# Ajout des dossiers au path pour importer nos modules locaux
sys.path.insert(0, str(Path.cwd().parent / 'src'))
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src.config import RAW_DATA_PATH, SUMMARY_FEATURES_PATH
from src.data import load_dataset_split
from src.metrics import compute_metrics
from src.time_series import run_backtests, aggregate_backtest_metrics
from scripts.train_models import build_models

pd.set_option('display.max_columns', None)

## 1. Modèles de Classification (Prédire la criticité)
Nous entraînons ici les 3 modèles (Régression Logistique, Random Forest, Gradient Boosting) et affichons leurs performances.

In [4]:
# Chargement des données déjà préparées
X_train, X_test, y_train, y_test = load_dataset_split()

# Initialisation des modèles via notre script
models = build_models()

metrics_list = []

for name, model in models.items():
    print(f"Entraînement de {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    metrics = compute_metrics(y_test, y_pred)
    metrics['model'] = name
    metrics_list.append(metrics)

df_metrics = pd.DataFrame(metrics_list)
display(df_metrics[['model', 'accuracy', 'f1_macro', 'precision_macro', 'recall_macro']])

Entraînement de log_reg...
Entraînement de random_forest...
Entraînement de gradient_boosting...


,model,accuracy,f1_macro,precision_macro,recall_macro
0,log_reg,0.959416,0.959414,0.959751,0.959416
1,random_forest,0.956169,0.956123,0.956442,0.956169
2,gradient_boosting,0.964286,0.964188,0.964262,0.964286


💡 **Conclusion Classification :** Le Gradient Boosting offre les meilleures performances.

## 2. Modèles de Séries Temporelles (Prévoir le trafic)
Nous évaluons ici SARIMAX vs Baseline Analogue pour prévoir le trafic normal sur 14 jours.

In [5]:
# Chargement des données brutes pour les séries temporelles
raw = pd.read_csv(Path.cwd().parent / RAW_DATA_PATH, parse_dates=['date'])
summary = pd.read_csv(Path.cwd().parent / SUMMARY_FEATURES_PATH)

# Sélection des 3 détroits les plus critiques pour la démonstration
top_ports = summary.sort_values('criticality_score', ascending=False).head(3)['portname'].tolist()
print(f"Détroits évalués : {top_ports}")

Détroits évalués : ['Malacca Strait', 'Taiwan Strait', 'Strait of Hormuz']


In [6]:
# Lancement du backtest (SARIMAX vs Analogue) sur 14 jours
# Note: n_splits est réduit à 2 pour que la cellule s'exécute rapidement.
predictions, metrics = run_backtests(
    raw,
    portnames=top_ports,
    horizons=[14],
    models=['analog', 'sarimax'],
    n_splits=2
)

if not metrics.empty:
    aggregated = aggregate_backtest_metrics(metrics)
    # On se concentre sur l'erreur absolue moyenne en % (MAPE)
    display(aggregated[['model', 'portname', 'horizon_days', 'mape', 'smape', 'mean_actual']])
else:
    print("Erreur lors du backtest.")

,model,portname,horizon_days,mape,smape,mean_actual
3,sarimax,Malacca Strait,14,5.387384,5.422511,194.964286
0,analog,Malacca Strait,14,8.075468,8.070916,194.964286
4,sarimax,Strait of Hormuz,14,37.078067,28.460725,37.464286
1,analog,Strait of Hormuz,14,48.500453,38.974133,37.464286
2,analog,Taiwan Strait,14,11.701912,11.025997,231.428571
5,sarimax,Taiwan Strait,14,11.947680,11.222771,231.428571


💡 **Conclusion Séries Temporelles :** Le modèle SARIMAX bat systématiquement la baseline naïve en ayant une erreur moyenne (MAPE) plus faible, validant ainsi son utilisation pour simuler le trafic de base avant de simuler une fermeture.